# validation check — KMeans (running) -> profitable clusters -> LR, DRAW market

`elo` space (2D: home vs away Club Elo rating), draw market. One segmentation method: KMeans on the running season
features, keeping the clusters that beat the market on the training fold. The k-sweep at the
bottom is an extra check, outside the template.

In [1]:
# --- SETUP -------------------------------------------------------------------
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

from evaluation.model_check import run_check, collect_bets, pooled_roi, roi_grouped, season_folds

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [2]:
# --- SPACE (modifiable) — the model space, defined in one place --------------
# The MODEL space: the features the bet-signal model reads. This is the ONLY cell to change
# to move the model to another space. (The segmentation space is set in the SEGMENTER cell.)
FEATURES = [
    "hmt_elo",
    "awt_elo",
]

In [3]:
# --- TARGET (modifiable) -----------------------------------------------------
# OUTCOME is the 0/1 event we bet on. IMPLIED is the market-consensus implied probability
# (mrkt), used for both the profitability check and the ROI — one odds source, never mixed.
OUTCOME = "t_draw_flg"          # 1 when the match ended in a draw
IMPLIED = "mrkt_draw_impl"      # implied draw prob = 1 / mrkt draw odds (profitability + betting)

In [4]:
# --- ABT FILTERING -----------------------------------------------------------
# Keep matches where both teams have enough history for stable season averages; drop rows missing any
# feature or the target.
MIN_GAME_NUMBER = 8
matches = abt[
    (abt["hmt_game_number"] > MIN_GAME_NUMBER)
    & (abt["awt_game_number"] > MIN_GAME_NUMBER)
].dropna(subset=FEATURES + [IMPLIED, OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 5533   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england', 'france', 'germany', 'italy', 'spain']


In [5]:
# --- SEGMENTER (modifiable) — KMeans on the running space --------------------
# SEGMENT_FEATURES is the space the SEGMENTATION uses. It defaults to the model's FEATURES (segment on
# the same variables the model sees); override it with a different list to segment on other variables
# than the model — e.g. carve regions by strength/context while the model still reads goals. The model
# space (FEATURES, used by run_check below) is unaffected.
SEGMENT_FEATURES = FEATURES

# The method IS this notebook's identity; a different method = a different notebook. Here: fit KMeans on
# SEGMENT_FEATURES (train only), keep the clusters that beat the market on train, label each row by its
# kept cluster (NaN outside -> not bet). Contract: segmenter(train) -> labeler ; labeler(rows) -> Series.

# Keep the clusters where the target happened MORE often than the market priced it, on the TRAIN fold
# only, so the held-out fold never influences the choice.
def keep_profitable(train, labels, min_matches=100):
    frame = pd.DataFrame({"label": labels.to_numpy(),
                          "won": train[OUTCOME].to_numpy(),
                          "implied": train[IMPLIED].to_numpy()})
    stats = frame.groupby("label").agg(won=("won", "mean"), implied=("implied", "mean"), n=("won", "size"))
    return set(stats.index[(stats["n"] >= min_matches) & (stats["won"] > stats["implied"])])

def kmeans_profitable_segmenter(features, n_clusters, min_matches=100):
    def segmenter(train):
        km = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(train[features].to_numpy())
        kept = keep_profitable(train, pd.Series(km.predict(train[features].to_numpy()), index=train.index), min_matches)
        def labeler(rows):
            c = pd.Series(km.predict(rows[features].to_numpy()), index=rows.index)
            return c.where(c.isin(kept))     # non-kept clusters -> NaN -> not bet
        return labeler
    return segmenter

K = 5   # cluster count (see the k-sweep extra check below)
segmenter = kmeans_profitable_segmenter(SEGMENT_FEATURES, K)

# The segmenter's self-description: fit on all matches (in-sample) and show each cluster's edge — size,
# outcome rate vs implied, ROI, centre (in SEGMENT_FEATURES). In-sample: a map of WHERE the edge sits,
# not gated evidence. KEEP marks clusters a training fold would keep (positive edge).
_km = KMeans(n_clusters=K, random_state=0, n_init=10).fit(matches[SEGMENT_FEATURES].to_numpy())
_lab = _km.predict(matches[SEGMENT_FEATURES].to_numpy())
print(f"segment edges (in-sample, k={K})   centre = [{'  '.join(SEGMENT_FEATURES)}]")
print(" seg      n   rate%  impl%   edge     ROI   keep   centre")
for s in range(K):
    m = matches[_lab == s]
    dr, im = m[OUTCOME].mean(), m[IMPLIED].mean()
    centre = " ".join(f"{x:.2f}" for x in _km.cluster_centers_[s])
    print(f" {s:>3} {len(m):>7}  {dr*100:5.1f}  {im*100:5.1f}  {(dr-im)*100:+5.1f}  "
          f"{pooled_roi(m[IMPLIED], m[OUTCOME].astype(int)):+6.3f}  {'KEEP' if dr > im else '    '}  [{centre}]")

segment edges (in-sample, k=5)   centre = [hmt_elo  awt_elo]
 seg      n   rate%  impl%   edge     ROI   keep   centre
   0    1217   24.8   27.7   -2.9  -0.105        [1615.81 1758.80]
   1    1303   22.9   24.2   -1.2  -0.050        [1758.05 1629.41]
   2    1373   30.7   29.2   +1.5  +0.052  KEEP  [1603.47 1599.62]
   3     750   18.5   20.7   -2.1  -0.103        [1911.71 1727.79]
   4     890   24.9   25.9   -1.0  -0.038        [1740.12 1890.74]


In [6]:
# --- MODEL (modifiable) ------------------------------------------------------
# The bet-signal model: any sklearn-style estimator with .fit(X, y) and .predict_proba(X). It is fit
# on the MODEL space (FEATURES) — independent of the segmentation space above.
def make_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                         StandardScaler(), LogisticRegression(max_iter=3000))

BUFFER = 0.02   # only bet when the model probability beats the implied one by at least this margin

## Summary
The three gates at a glance, and the one-line row written to `model_checks/results.md`.

In [7]:
_ = run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=segmenter, make_model=make_model, buffer=BUFFER,
              source="elo_draw_kmeans_match_lr", space="elo", target="draw",
              abt_filter="major, gn>8, dropna",
              segmentation=f"KMeans k={K} (match, running, {len(SEGMENT_FEATURES)}D)",
              segment_filter="train-profitable clusters")

elo_draw_kmeans_match_lr   pooled +0.007 (592 bets)   [mrkt]

GATE 1  LOSO — ROI by test season           PASS  (+3/4 seasons)
    2223 +0.119    2324 +0.188    2425 -0.270    2526 +0.130

GATE 2  league — ROI by league / drop best   FAIL
    italy +0.323    germany +0.046    france -0.037    spain -0.148    england -0.222
    pooled without best league (italy): -0.046

GATE 3  walk-forward — ROI by test season    FAIL
    2425 -0.135    2526 +0.130


## Extra checks — below the template
Everything above is the standard template (config cells + Summary). Cells below are notebook-specific
diagnostics, outside the template — they may use knobs a given segmenter has (e.g. a cluster count)
that other methods do not.

In [8]:
# EXTRA CHECK (below the template) — knob robustness: sweep the cluster count k = 2..8.
# Recomputes the whole honest per-fold pipeline for each k (segmenting on SEGMENT_FEATURES, model on
# FEATURES). A real edge survives a range of k; one that appears at a single k is fragile.
print(" k    pooled     bets   seasons+")
for k in range(2, 9):
    b = collect_bets(matches, season_folds(matches), features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
                     segmenter=kmeans_profitable_segmenter(SEGMENT_FEATURES, k), make_model=make_model, buffer=BUFFER)
    ps = roi_grouped(b, "fold")
    print(f" {k:<3} {pooled_roi(b['implied'], b['won']):+.4f}   {len(b):>4}   +{int((ps > 0).sum())}/{len(ps)}")

 k    pooled     bets   seasons+


 2   -0.2518    294   +0/1


 3   -0.2746    301   +0/2


 4   -0.0827    477   +1/3


 5   +0.0072    592   +3/4


 6   +0.0122    556   +3/4


 7   -0.0676    706   +2/4


 8   -0.0640    724   +2/4
